<a href="https://colab.research.google.com/github/deepa22-eng/Machine-Learning-2026/blob/main/Deepa_Assignment_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Chemical Applications of Machine Learning (CHEM 4930/5610) - Spring 2026

### Assignment 3 - Deadline 2/27/2026
Points 10

#### General Comments
All figures and graph should have approriate labels on the two axis, and should include a legend with appropriate labels of the different plots.

The notebook should be return in working format. That is, I should be able to reset all the output and re-run all the cells and get the same results as you obtained.

**You should start by saving a copy of the notebook to your Google Drive so you preserve all changes.**

**Please add your name as a suffix to the filname**

**Student Name**: Deepa Ranabhat

**AI usage statement:**
I used Claude AI for understanding and interpretation of the code that I used to run the calculation.

### Task 1 - 10 points

In this task, we will consider data from this paper:
- Enhancing Permeability Prediction of Heterobifunctional Degraders Using Machine Learning and Metadynamics-Informed 3D Molecular Descriptors - [DOI:10.1021/acs.jcim.5c01600](https://doi.org/10.1021/acs.jcim.5c01600)

Where the authors consider the Permeability of so-called PROTAC compounds that are large and flexible molecules used in Targeted Protein Degradation.

All the dataset used in the paper, and the code use to obtain the results are given in this following Github repository:
- https://github.com/brykimjh/degrader-permeability-ml3d-metaD  

The specfic dataset that we use 32 PROTAC compounds with measured passive permeability (given in nm/s) and includes 17 2D features calculated by RDKit (see [here](https://github.com/brykimjh/degrader-permeability-ml3d-metaD/blob/main/data/calculate_2d_properties.py) for the script they are calculated), and 3 3D "ensemble" features that are obtained from molecular dynamics simulations as described in the paper.

The target value is the measured passive permeability that is experimentaly measured and given in nm/s.

The dataset can be seen here:
- https://github.com/brykimjh/degrader-permeability-ml3d-metaD/blob/main/outputs/ml_models/model_data.csv

Where the log10 transformed passive permeability is given by `P_appLog`. Note that here the passive permeability has been log transformed in based 10 using `np.log10`.

The 2D features obtained using RDKit are:
```
[
 'Molecular Weight (MW)', 'CharVol (characteristic volume)',
 'Flexibility (number of rotatable bonds / number of bonds)',
 'Number of Heavy Atoms (HA)', 'RingAtoms', 'Halogens', 'HeteroAtoms',
 'RotBonds (NRotB)', 'AllBonds', 'RingCount', 'NumStereo',
 'Fraction of sp3 Carbon Atoms (FSP3)', 'Hydrogen Bond Donors (HBD)',
 'Hydrogen Bond Acceptors (HBA)', 'cLogD^7.4',
 'Topological polar surface area (TPSA)',
 'Total non-polar surface area (TNSA)'
]
```
that includes various standard descriptors/features implemented in RDKit

and the 3D "ensemble" features obtained using MD simulations are
```
[
 'Ensemble_Average_PSA_Chloroform_ANI',
 'Ensemble_Average_Num_IMHB_Chloroform_ANI',
 'Ensemble_Average_RadiusOfGyration_Chloroform_ANI'
]
```
which inludes the average polar surface area (PSA), the average number of intramolecular hydrogen bonds (IMHB), and the average radius of gyration (that measures if the molecule is compact or extended).

#### A)
Split the dataset into compounds with two classes with low and high permeability evenly so each class has 16 compounds.

#### B)
For all 32 compounds, visualize the 2D chemical structure using RDKit and the mols2grid package, and show their measured passive permeability in nm/s and their low/high permeability classifciation.

#### C)
Perform classifcation using random forest classification model using the following hyperparameters:
- Number of tree: 100
- Maximum depth of each tree: 2
- Maximum number of features for each tree: 50% of the number of input features (rounded up if that number is a fraction)

You should perform the classification using 3 different feature set:
- 2D RDKit features
- 3D Ensemble features
- Combined set of 2D and 3D features

For each case, perform cross validation (CV) using a CV strategy of your choice, and obtain the average and standard deviation of metrics that measure the performance, including:
- Accuracy
- Precision
- Receiver Operating Characteristic Area Under the Curve (ROC AUC)  

In your analysis and metrics, the high permeability class should be consider as the postive class.

Based on your analysis, which feature set gives the best results for the classifcation?

#### D) - Optional for 2 points
Repeat your analysis from C) using a $k$ nearest neighbors classifer (i.e., $k$ nearest neighbors vote). Use the default value of 5 neighbors.

Do you obtain the same results as in C)?



In [106]:
# Bash script to download all the dataset. Don't worry if you don't understand it
%%bash

url="https://raw.githubusercontent.com/brykimjh/degrader-permeability-ml3d-metaD/refs/heads/main/outputs/ml_models/"
dataset_filename="model_data.csv"

rm -f ${dataset_filename}

wget ${url}/${dataset_filename} &> /dev/null

ls

2d_features.csv
model_data.csv
sample_data


In [107]:
# Bash script to download all the dataset. Don't worry if you don't understand it
%%bash

url="https://raw.githubusercontent.com/brykimjh/degrader-permeability-ml3d-metaD/refs/heads/main/outputs/ml_models/"
dataset_filename="model_data.csv"

rm -f ${dataset_filename}

wget ${url}/${dataset_filename} &> /dev/null

ls

2d_features.csv
model_data.csv
sample_data


In [108]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [109]:
dataset_1 = pd.read_csv("model_data.csv")
dataset_2 = pd.read_csv("2d_features.csv")

In [110]:
dataset_1


,Molecular Weight (MW),CharVol (characteristic volume),Flexibility (number of rotatable bonds / number of bonds),Number of Heavy Atoms (HA),RingAtoms,Halogens,HeteroAtoms,RotBonds (NRotB),AllBonds,RingCount,...,Fraction of sp3 Carbon Atoms (FSP3),Hydrogen Bond Donors (HBD),Hydrogen Bond Acceptors (HBA),cLogD^7.4,Topological polar surface area (TPSA),Total non-polar surface area (TNSA),Ensemble_Average_PSA_Chloroform_ANI,Ensemble_Average_Num_IMHB_Chloroform_ANI,Ensemble_Average_RadiusOfGyration_Chloroform_ANI,P_appLog
0,896.999,803.360,0.253521,65,42,0,19,18,71,7,...,0.456522,3,14,2.64400,209.98,710.02,193.71,0.97,5.68,1.491362
1,852.902,743.496,0.191176,62,42,0,19,13,68,7,...,0.395349,3,13,1.76390,217.82,642.18,252.14,0.87,5.24,1.049218
2,850.930,753.592,0.191176,62,42,0,18,13,68,7,...,0.409091,3,12,2.45750,208.59,671.41,248.01,0.67,5.47,0.812913
3,1020.210,896.016,0.276316,71,33,3,20,21,76,6,...,0.470588,3,12,7.70720,186.66,833.34,186.01,0.60,5.84,0.462398
4,1012.704,900.176,0.236842,70,39,1,20,18,76,7,...,0.460000,5,14,7.42946,214.98,785.02,105.63,2.92,5.60,-0.397940
5,837.830,697.696,0.265625,59,34,3,19,17,64,6,...,0.375000,1,12,4.40258,177.04,622.96,204.17,0.29,5.23,1.690196
6,1006.183,872.816,0.266667,70,33,3,20,20,75,6,...,0.460000,3,12,7.31710,186.66,813.34,175.16,0.32,5.41,1.136721
7,1128.150,931.232,0.256098,77,33,9,26,21,82,6,...,0.470588,3,12,8.44280,186.66,833.34,160.07,0.93,5.93,0.505150
8,1022.182,891.056,0.276316,71,33,3,21,21,76,6,...,0.460000,3,13,6.55350,195.89,804.11,197.66,1.64,5.16,1.235528
9,1078.143,902.096,0.253165,74,33,7,24,20,79,6,...,0.460000,3,12,7.80750,186.66,813.34,217.11,0.00,5.63,0.113943


In [111]:
dataset_2

,Index,Compound,Smiles,P_app AB (nm/s),P_app BA (nm/s),P_app,Molecular Weight (MW),CharVol (characteristic volume),Flexibility (number of rotatable bonds / number of bonds),Number of Heavy Atoms (HA),...,RotBonds (NRotB),AllBonds,RingCount,NumStereo,Fraction of sp3 Carbon Atoms (FSP3),Hydrogen Bond Donors (HBD),Hydrogen Bond Acceptors (HBA),cLogD^7.4,Topological polar surface area (TPSA),Total non-polar surface area (TNSA)
0,1,6a,O=C(C(N1C(C2=CC=CC(NCCOCCOCCOCCC(N(CC3)CCN3C(C...,2.6,370,31.016125,896.999,803.360,0.253521,65.0,...,18.0,71.0,7.0,2.0,0.456522,3.0,14.0,2.64400,209.98,710.02
1,2,6b,O=C(C(N1C(C2=CC=CC(NC(COCCOCC(N(CC3)CCN3C(C=C4...,1.3,96,11.171392,852.902,743.496,0.191176,62.0,...,13.0,68.0,7.0,2.0,0.395349,3.0,13.0,1.76390,217.82,642.18
2,3,6c,O=C(C(N1C(C2=CC=CC(OCC(NCCCCC(N(CC3)CCN3C(C=C4...,1.0,42,6.480741,850.930,753.592,0.191176,62.0,...,13.0,68.0,7.0,2.0,0.409091,3.0,12.0,2.45750,208.59,671.41
3,4,2d,CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)...,3.5,2,2.900000,1020.210,896.016,0.276316,71.0,...,21.0,76.0,6.0,3.0,0.470588,3.0,12.0,7.70720,186.66,833.34
4,5,4a,CC1=NC(NC2=NC=C(C(=O)NC3=C(C)C=CC=C3Cl)S2)=CC(...,0.2,1,0.400000,1012.704,900.176,0.236842,70.0,...,18.0,76.0,7.0,3.0,0.460000,5.0,14.0,7.42946,214.98,785.02
5,6,2f,CC1(C)C(=O)N(C2=CC=C(C#N)C(C(F)(F)F)=C2)C(=S)N...,17.0,141,49.000000,837.830,697.696,0.265625,59.0,...,17.0,64.0,6.0,1.0,0.375000,1.0,12.0,4.40258,177.04,622.96
6,7,2b,CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)...,13.5,14,13.700000,1006.183,872.816,0.266667,70.0,...,20.0,75.0,6.0,3.0,0.460000,3.0,12.0,7.31710,186.66,813.34
7,8,2c,CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)...,2.6,4,3.200000,1128.150,931.232,0.256098,77.0,...,21.0,82.0,6.0,3.0,0.470588,3.0,12.0,8.44280,186.66,833.34
8,9,2a,CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)...,3.4,86,17.200000,1022.182,891.056,0.276316,71.0,...,21.0,76.0,6.0,3.0,0.460000,3.0,13.0,6.55350,195.89,804.11
9,10,2e,CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)...,0.6,3,1.300000,1078.143,902.096,0.253165,74.0,...,20.0,79.0,6.0,3.0,0.460000,3.0,12.0,7.80750,186.66,813.34


In [112]:
dataset_1['P_app'] = dataset_2['P_app']
dataset_1['Smiles'] = dataset_2['Smiles']

In [113]:
dataset_1

,Molecular Weight (MW),CharVol (characteristic volume),Flexibility (number of rotatable bonds / number of bonds),Number of Heavy Atoms (HA),RingAtoms,Halogens,HeteroAtoms,RotBonds (NRotB),AllBonds,RingCount,...,Hydrogen Bond Acceptors (HBA),cLogD^7.4,Topological polar surface area (TPSA),Total non-polar surface area (TNSA),Ensemble_Average_PSA_Chloroform_ANI,Ensemble_Average_Num_IMHB_Chloroform_ANI,Ensemble_Average_RadiusOfGyration_Chloroform_ANI,P_appLog,P_app,Smiles
0,896.999,803.360,0.253521,65,42,0,19,18,71,7,...,14,2.64400,209.98,710.02,193.71,0.97,5.68,1.491362,31.016125,O=C(C(N1C(C2=CC=CC(NCCOCCOCCOCCC(N(CC3)CCN3C(C...
1,852.902,743.496,0.191176,62,42,0,19,13,68,7,...,13,1.76390,217.82,642.18,252.14,0.87,5.24,1.049218,11.171392,O=C(C(N1C(C2=CC=CC(NC(COCCOCC(N(CC3)CCN3C(C=C4...
2,850.930,753.592,0.191176,62,42,0,18,13,68,7,...,12,2.45750,208.59,671.41,248.01,0.67,5.47,0.812913,6.480741,O=C(C(N1C(C2=CC=CC(OCC(NCCCCC(N(CC3)CCN3C(C=C4...
3,1020.210,896.016,0.276316,71,33,3,20,21,76,6,...,12,7.70720,186.66,833.34,186.01,0.60,5.84,0.462398,2.900000,CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)...
4,1012.704,900.176,0.236842,70,39,1,20,18,76,7,...,14,7.42946,214.98,785.02,105.63,2.92,5.60,-0.397940,0.400000,CC1=NC(NC2=NC=C(C(=O)NC3=C(C)C=CC=C3Cl)S2)=CC(...
5,837.830,697.696,0.265625,59,34,3,19,17,64,6,...,12,4.40258,177.04,622.96,204.17,0.29,5.23,1.690196,49.000000,CC1(C)C(=O)N(C2=CC=C(C#N)C(C(F)(F)F)=C2)C(=S)N...
6,1006.183,872.816,0.266667,70,33,3,20,20,75,6,...,12,7.31710,186.66,813.34,175.16,0.32,5.41,1.136721,13.700000,CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)...
7,1128.150,931.232,0.256098,77,33,9,26,21,82,6,...,12,8.44280,186.66,833.34,160.07,0.93,5.93,0.505150,3.200000,CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)...
8,1022.182,891.056,0.276316,71,33,3,21,21,76,6,...,13,6.55350,195.89,804.11,197.66,1.64,5.16,1.235528,17.200000,CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)...
9,1078.143,902.096,0.253165,74,33,7,24,20,79,6,...,12,7.80750,186.66,813.34,217.11,0.00,5.63,0.113943,1.300000,CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)...


A) Splitting the dataset into compounds with two classes with low and high permeability evenly so each class has 16 compounds.

In [114]:
Permeable_cutoff = (7)
Low_label = 0
High_label = +1
Permeable_key_str = f'Permeability High({High_label:})/Low({Low_label:})'
dataset_1[Permeable_key_str] = [High_label if p > Permeable_cutoff else Low_label for p in dataset_1['P_app']]

Number_Permeable_High = np.sum(dataset_1[Permeable_key_str] == +1)
Number_Permeable_Low = np.sum(dataset_1[Permeable_key_str] == 0)

print("Key:",Permeable_key_str)

print("Number with high permeability (above {:.1f} nm/s): {:d}".format(Permeable_cutoff,Number_Permeable_High))
print("Number with low permeability (above {:.1f} nm/s): {:d}".format(Permeable_cutoff,Number_Permeable_Low))

print("")

dataset_1[['P_app', Permeable_key_str] ]

Key: Permeability High(1)/Low(0)
Number with high permeability (above 7.0 nm/s): 16
Number with low permeability (above 7.0 nm/s): 16



,P_app,Permeability High(1)/Low(0)
0,31.016125,1
1,11.171392,1
2,6.480741,0
3,2.900000,0
4,0.400000,0
5,49.000000,1
6,13.700000,1
7,3.200000,0
8,17.200000,1
9,1.300000,0


B) For all 32 compounds, visualize the 2D chemical structure using RDKit and the mols2grid package, and show their measured passive permeability in nm/s and their low/high permeability classifciation.

In [115]:
# the %%capture command will surpress output to screen
%%capture
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install rdkit mols2grid

In [116]:
import mols2grid


In [117]:
dataset_1['P_app(nm/s)'] = dataset_1['P_app'].round(2)

In [118]:
# by passing the dataframe, and giving the column with the SMILES string, we can
# plot all molecules
# we can also add information to the figures by using the subset variable
mols2grid.display(dataset_1,smiles_col='Smiles',subset=["img", 'P_app(nm/s)' ,Permeable_key_str ])

C) Perform classifcation using random forest classification model using the following hyperparameters:

Number of tree: 100
Maximum depth of each tree: 2
Maximum number of features for each tree: 50% of the number of input features (rounded up if that number is a fraction)
You should perform the classification using 3 different feature set:

2D RDKit features
3D Ensemble features
Combined set of 2D and 3D features
For each case, perform cross validation (CV) using a CV strategy of your choice, and obtain the average and standard deviation of metrics that measure the performance, including:

Accuracy
Precision
Receiver Operating Characteristic Area Under the Curve (ROC AUC)
In your analysis and metrics, the high permeability class should be consider as the postive class.



In [119]:
import math
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold, ShuffleSplit
from sklearn import metrics

In [120]:
features_2d = [
    'Molecular Weight (MW)',
    'CharVol (characteristic volume)',
    'Flexibility (number of rotatable bonds / number of bonds)',
    'Number of Heavy Atoms (HA)',
    'RingAtoms',
    'Halogens',
    'HeteroAtoms',
    'RotBonds (NRotB)',
    'AllBonds',
    'RingCount',
    'NumStereo',
    'Fraction of sp3 Carbon Atoms (FSP3)',
    'Hydrogen Bond Donors (HBD)',
    'Hydrogen Bond Acceptors (HBA)',
    'cLogD^7.4',
    'Topological polar surface area (TPSA)',
    'Total non-polar surface area (TNSA)'
]

features_3d = [
    'Ensemble_Average_PSA_Chloroform_ANI',
    'Ensemble_Average_Num_IMHB_Chloroform_ANI',
    'Ensemble_Average_RadiusOfGyration_Chloroform_ANI'
]

features_combined = features_2d + features_3d

For 2D-Features

In [ ]:
import math
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold, ShuffleSplit
from sklearn import metrics

# Features and target
features = dataset_1[features_2d].values
target   = dataset_1[Permeable_key_str].values

# max_features = 50% rounded up
n_max_features = math.ceil(len(features_2d) * 0.5)
print(f"Number of 2D features: {len(features_2d)}")
print(f"max_features: {n_max_features}")

# Model
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=2,
    max_features=n_max_features,
    random_state=42
)

# Scoring
scoring = {
    'accuracy' : 'accuracy',
    'precision': metrics.make_scorer(metrics.precision_score, zero_division=np.nan),
    'roc_auc'  : 'roc_auc'
}

# CV strategies
NumSplits = 100
scores_fold   = cross_validate(model, features, target,
                                scoring=scoring,
                                cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42))

scores_random = cross_validate(model, features, target,
                                scoring=scoring,
                                cv=ShuffleSplit(n_splits=NumSplits, test_size=0.5))

# Print results
print("\n========== 2D RDKit Features ==========")
print("Accuracy")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(scores_fold['test_accuracy'].mean(),   scores_fold['test_accuracy'].std()))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(scores_random['test_accuracy'].mean(), scores_random['test_accuracy'].std()))

print("Precision")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(np.nanmean(scores_fold['test_precision']),   np.nanstd(scores_fold['test_precision'])))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(np.nanmean(scores_random['test_precision']), np.nanstd(scores_random['test_precision'])))

print("ROC AUC")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(scores_fold['test_roc_auc'].mean(),   scores_fold['test_roc_auc'].std()))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(scores_random['test_roc_auc'].mean(), scores_random['test_roc_auc'].std()))

Number of 2D features: 17
max_features: 9


For 3D-features

In [ ]:
# Features and target
features = dataset_1[features_3d].values
target   = dataset_1[Permeable_key_str].values

# max_features = 50% rounded up
n_max_features = math.ceil(len(features_3d) * 0.5)
print(f"Number of 3D features: {len(features_3d)}")
print(f"max_features: {n_max_features}")

# Model
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=2,
    max_features=n_max_features,
    random_state=42
)

# Scoring
scoring = {
    'accuracy' : 'accuracy',
    'precision': metrics.make_scorer(metrics.precision_score, zero_division=np.nan),
    'roc_auc'  : 'roc_auc'
}

# CV strategies
NumSplits = 100
scores_fold   = cross_validate(model, features, target,
                                scoring=scoring,
                                cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42))

scores_random = cross_validate(model, features, target,
                                scoring=scoring,
                                cv=ShuffleSplit(n_splits=NumSplits, test_size=0.5))

# Print results
print("\n========== 3D RDKit Features ==========")
print("Accuracy")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(scores_fold['test_accuracy'].mean(),   scores_fold['test_accuracy'].std()))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(scores_random['test_accuracy'].mean(), scores_random['test_accuracy'].std()))

print("Precision")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(np.nanmean(scores_fold['test_precision']),   np.nanstd(scores_fold['test_precision'])))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(np.nanmean(scores_random['test_precision']), np.nanstd(scores_random['test_precision'])))

print("ROC AUC")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(scores_fold['test_roc_auc'].mean(),   scores_fold['test_roc_auc'].std()))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(scores_random['test_roc_auc'].mean(), scores_random['test_roc_auc'].std()))

For 2D and 3D combined features

In [ ]:
# Features and target
features = dataset_1[features_combined].values
target   = dataset_1[Permeable_key_str].values

# max_features = 50% rounded up
n_max_features = math.ceil(len(features_combined) * 0.5)
print(f"Number of combined features: {len(features_combined)}")
print(f"max_features: {n_max_features}")

# Model
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=2,
    max_features=n_max_features,
    random_state=42
)

# Scoring
scoring = {
    'accuracy' : 'accuracy',
    'precision': metrics.make_scorer(metrics.precision_score, zero_division=np.nan),
    'roc_auc'  : 'roc_auc'
}

# CV strategies
NumSplits = 100
scores_fold   = cross_validate(model, features, target,
                                scoring=scoring,
                                cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42))

scores_random = cross_validate(model, features, target,
                                scoring=scoring,
                                cv=ShuffleSplit(n_splits=NumSplits, test_size=0.5))

# Print results
print("\n========== Combined RDKit Features ==========")
print("Accuracy")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(scores_fold['test_accuracy'].mean(),   scores_fold['test_accuracy'].std()))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(scores_random['test_accuracy'].mean(), scores_random['test_accuracy'].std()))

print("Precision")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(np.nanmean(scores_fold['test_precision']),   np.nanstd(scores_fold['test_precision'])))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(np.nanmean(scores_random['test_precision']), np.nanstd(scores_random['test_precision'])))

print("ROC AUC")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(scores_fold['test_roc_auc'].mean(),   scores_fold['test_roc_auc'].std()))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(scores_random['test_roc_auc'].mean(), scores_random['test_roc_auc'].std()))

The 3D ensembles features gives the best calculation result among three. The 3D ensemble features (PSA, IMHB, radius of gyration)
obtained from MD simulations are the most informative for
predicting passive permeability. This makes physical sense
because permeability depends on the 3D conformation and
dynamic behavior of the molecule in a membrane-like
environment, which the 2D RDKit descriptors cannot capture.

Interestingly, adding 2D features to the 3D features
(combined) actually slightly decreased performance compared
to 3D alone, suggesting the 2D features may add noise
rather than useful information for this small dataset of
32 compounds.

D) Optional
 Analysing a  𝑘  nearest neighbors classifer (i.e.,  𝑘  nearest neighbors vote) by using the default value of 5 neighbors.


For 2D- features


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# Features and target
features = dataset_1[features_2d].values
target   = dataset_1[Permeable_key_str].values

# KNN model with StandardScaler
model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("knn",    KNeighborsClassifier(n_neighbors=5))
])

# Scoring
scoring = {
    'accuracy' : 'accuracy',
    'precision': metrics.make_scorer(metrics.precision_score,
                                     pos_label=1,
                                     zero_division=np.nan),
    'roc_auc'  : 'roc_auc'
}

# CV strategies
NumSplits = 100
scores_fold   = cross_validate(model, features, target,
                                scoring=scoring,
                                cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42))

scores_random = cross_validate(model, features, target,
                                scoring=scoring,
                                cv=ShuffleSplit(n_splits=NumSplits, test_size=0.5))

# Print results
print("\n========== KNN - 2D RDKit Features ==========")
print("Accuracy")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(scores_fold['test_accuracy'].mean(),   scores_fold['test_accuracy'].std()))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(scores_random['test_accuracy'].mean(), scores_random['test_accuracy'].std()))

print("Precision")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(np.nanmean(scores_fold['test_precision']),   np.nanstd(scores_fold['test_precision'])))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(np.nanmean(scores_random['test_precision']), np.nanstd(scores_random['test_precision'])))

print("ROC AUC")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(scores_fold['test_roc_auc'].mean(),   scores_fold['test_roc_auc'].std()))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(scores_random['test_roc_auc'].mean(), scores_random['test_roc_auc'].std()))

For 3D- features

In [ ]:


# Features and target
features = dataset_1[features_3d].values
target   = dataset_1[Permeable_key_str].values

# KNN model with StandardScaler
model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("knn",    KNeighborsClassifier(n_neighbors=5))
])

# Scoring
scoring = {
    'accuracy' : 'accuracy',
    'precision': metrics.make_scorer(metrics.precision_score,
                                     pos_label=1,
                                     zero_division=np.nan),
    'roc_auc'  : 'roc_auc'
}

# CV strategies
NumSplits = 100
scores_fold   = cross_validate(model, features, target,
                                scoring=scoring,
                                cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42))

scores_random = cross_validate(model, features, target,
                                scoring=scoring,
                                cv=ShuffleSplit(n_splits=NumSplits, test_size=0.5))

# Print results
print("\n========== KNN - 3D RDKit Features ==========")
print("Accuracy")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(scores_fold['test_accuracy'].mean(),   scores_fold['test_accuracy'].std()))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(scores_random['test_accuracy'].mean(), scores_random['test_accuracy'].std()))

print("Precision")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(np.nanmean(scores_fold['test_precision']),   np.nanstd(scores_fold['test_precision'])))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(np.nanmean(scores_random['test_precision']), np.nanstd(scores_random['test_precision'])))

print("ROC AUC")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(scores_fold['test_roc_auc'].mean(),   scores_fold['test_roc_auc'].std()))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(scores_random['test_roc_auc'].mean(), scores_random['test_roc_auc'].std()))

For 2D and 3D combined features.

In [ ]:


# Features and target
features = dataset_1[features_combined].values
target   = dataset_1[Permeable_key_str].values

# KNN model with StandardScaler
model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("knn",    KNeighborsClassifier(n_neighbors=5))
])

# Scoring
scoring = {
    'accuracy' : 'accuracy',
    'precision': metrics.make_scorer(metrics.precision_score,
                                     pos_label=1,
                                     zero_division=np.nan),
    'roc_auc'  : 'roc_auc'
}

# CV strategies
NumSplits = 100
scores_fold   = cross_validate(model, features, target,
                                scoring=scoring,
                                cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42))

scores_random = cross_validate(model, features, target,
                                scoring=scoring,
                                cv=ShuffleSplit(n_splits=NumSplits, test_size=0.5))

# Print results
print("\n========== KNN - Combined RDKit Features ==========")
print("Accuracy")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(scores_fold['test_accuracy'].mean(),   scores_fold['test_accuracy'].std()))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(scores_random['test_accuracy'].mean(), scores_random['test_accuracy'].std()))

print("Precision")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(np.nanmean(scores_fold['test_precision']),   np.nanstd(scores_fold['test_precision'])))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(np.nanmean(scores_random['test_precision']), np.nanstd(scores_random['test_precision'])))

print("ROC AUC")
print("  5-Fold CV          : {:.3f} +/- {:.3f}".format(scores_fold['test_roc_auc'].mean(),   scores_fold['test_roc_auc'].std()))
print("  Random Splits(100) : {:.3f} +/- {:.3f}".format(scores_random['test_roc_auc'].mean(), scores_random['test_roc_auc'].std()))

In KNN , I cant get same result as in random forest. However,  3D- ensembles gives the best reult among three like as in random forest analyis.
If we compare both model the random forest ROU value is higher than KNN model. So for 32- compound data set random forest model seem work well.